# SG Alerts — Checkpoint Vision Distillation (Colab GPU → Free CPU ONNX)

Use your monthly Google Colab GPU compute units **once** (~10–20 units) instead of paying for a 24/7 GPU:
1. **Collect** real Woodlands (`2701`, `2702`) and Tuas (`4703`, `4713`) frames across day, night, rain, and contraflow.
2. **Auto-label (Teacher)** on Colab GPU using high-recall tiled detection (and optionally `omlab/VLX-Seek` for hard night/rain/contraflow verification).
3. **Fine-tune (Student)** a compact `YOLO` / `YOLOX` detector specifically on these 4 fixed Singapore camera angles.
4. **Export `sgalerts_custom.onnx`** and place it at `.cache/models/sgalerts_custom.onnx` — `scripts/analyze-checkpoints.py` will load it automatically on GitHub Actions CPU for $0/month.

In [ ]:
# 1. Check GPU & install lightweight distillation tools
!nvidia-smi
!pip -q install requests pillow onnx onnxruntime ultralytics sahi

In [ ]:
# 2. Collect live snapshots from LTA data.gov.sg (2701, 2702, 4703, 4713)
# Tip: run this cell periodically or upload frames saved locally via:
#   python scripts/analyze-checkpoints.py --save-samples samples/
import io, time, requests
from pathlib import Path
from PIL import Image

CAM_IDS = {"2701", "2702", "4703", "4713"}
RAW_DIR = Path("/content/sgalerts_dataset/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def grab_once():
    r = requests.get("https://api.data.gov.sg/v1/transport/traffic-images", timeout=20)
    r.raise_for_status()
    cams = r.json()["items"][0]["cameras"]
    ts = int(time.time())
    saved = 0
    for cam in cams:
        cid = str(cam.get("camera_id", ""))
        if cid in CAM_IDS:
            img_bytes = requests.get(cam["image"], timeout=20).content
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            img.save(RAW_DIR / f"cam_{cid}_{ts}.jpg", quality=95)
            saved += 1
    print(f"Saved {saved} frames to {RAW_DIR} (total: {len(list(RAW_DIR.glob('*.jpg')))})")

grab_once()

In [ ]:
# 3. Auto-label raw frames using high-accuracy SAHI tiled GPU Teacher
# Maps COCO vehicle classes -> 0:car, 1:motorcycle, 2:bus, 3:truck
from pathlib import Path
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

COCO_TO_SGA = {2: 0, 3: 1, 5: 2, 7: 3}
DATASET_DIR = Path("/content/sgalerts_dataset")
IMG_TRAIN = DATASET_DIR / "images/train"
LBL_TRAIN = DATASET_DIR / "labels/train"
IMG_TRAIN.mkdir(parents=True, exist_ok=True)
LBL_TRAIN.mkdir(parents=True, exist_ok=True)

teacher = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolo11x.pt",
    confidence_threshold=0.22,
    device="cuda:0",
)

for img_path in sorted((DATASET_DIR / "raw").glob("*.jpg")):
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    res = get_sliced_prediction(
        str(img_path),
        teacher,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.25,
        overlap_width_ratio=0.25,
    )
    lines = []
    for pred in res.object_prediction_list:
        cid = pred.category.id
        if cid not in COCO_TO_SGA:
            continue
        x1, y1, x2, y2 = pred.bbox.to_xyxy()
        cx = ((x1 + x2) / 2) / W
        cy = ((y1 + y2) / 2) / H
        bw = (x2 - x1) / W
        bh = (y2 - y1) / H
        lines.append(f"{COCO_TO_SGA[cid]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    im.save(IMG_TRAIN / img_path.name, quality=95)
    (LBL_TRAIN / f"{img_path.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    print(f"{img_path.name}: {len(lines)} vehicles labeled")

### Optional: Verify Hard Night / Rain / Contraflow Frames with `VLX-Seek`
If you want to use **`om-ai-lab/VLX-Seek`** (`omlab/VLX-Seek-1.5-10B` on Colab L4/A100) to double-check ambiguous night tail-lights or Tuas orange contraflow cones, clone `https://github.com/om-ai-lab/VLX-Seek` and query your hardest frames to refine labels before training the student below.

In [ ]:
# 4. Fine-tune compact Student detector & export ONNX
# NOTE: YOLOX-format ONNX drops straight into scripts/analyze-checkpoints.py
# via .cache/models/sgalerts_custom.onnx or SGALERTS_CUSTOM_ONNX env var.
from pathlib import Path
from ultralytics import YOLO

yaml_path = Path("/content/sgalerts_dataset/data.yaml")
yaml_path.write_text(
    "path: /content/sgalerts_dataset\n"
    "train: images/train\n"
    "val: images/train\n"
    "names:\n"
    "  0: car\n"
    "  1: motorcycle\n"
    "  2: bus\n"
    "  3: truck\n",
    encoding="utf-8",
)

student = YOLO("yolo11s.pt")
student.train(data=str(yaml_path), epochs=30, imgsz=640, batch=16)
onnx_out = student.export(format="onnx", simplify=True, imgsz=640)
print("Exported distilled student ONNX:", onnx_out)